In [ ]:
# 1. Mount your Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Define cloud paths (Now safely wrapped in quotes)
import os
import shutil

zip_path = '/content/drive/MyDrive/PhD/Metahuman_project/fmetahuman_data_parsed.zip'
extract_destination = '/content/fmetahuman_data/'

# 3. Unzip the dataset to the local cloud environment
if os.path.exists(zip_path):
    print("Extracting dataset to local storage for high-speed I/O...")
    shutil.unpack_archive(zip_path, extract_destination)
    print(f"Extraction complete. Data ready at: {extract_destination}")
else:
    print(f"Error: Could not find the zip file at {zip_path}. Please check your Drive path.")

Mounted at /content/drive
Extracting dataset to local storage for high-speed I/O...
Extraction complete. Data ready at: /content/fmetahuman_data/


In [ ]:
import os

def locate_metahuman_zip(base_path, target_name):
    print(f"Scanning Google Drive for '{target_name}'... Please wait.")
    for root, dirs, files in os.walk(base_path):
        # Check for exact match or common Windows double-extension issue (.zip.zip)
        for file in files:
            if file.lower() == target_name.lower() or file.lower() == (target_name + ".zip").lower():
                exact_path = os.path.join(root, file)
                return exact_path
    return None

# Trigger the search across your mounted Drive space
found_path = locate_metahuman_zip('/content/drive/MyDrive/', 'fmetahuman_data_parsed.zip')

if found_path:
    print("\n" + "="*60)
    print("SUCCESS! Found your file.")
    print(f"Copy this exact path:\n\n{found_path}")
    print("="*60)
else:
    print("\n" + "="*60)
    print("Could not find the file automatically.")
    print("Double-check that the file finished uploading completely to Drive,")
    print("and that it is named exactly 'fmetahuman_data_parsed.zip'.")
    print("="*60)


Scanning Google Drive for 'fmetahuman_data_parsed.zip'... Please wait.

SUCCESS! Found your file.
Copy this exact path:

/content/drive/MyDrive/PhD/Metahuman_project/fmetahuman_data_parsed.zip


In [ ]:
import os

base_path = '/content/fmetahuman_data/'
if os.path.exists(base_path):
    print("Folders found inside metahuman_data:")
    print(os.listdir(base_path))
else:
    print("The directory does not exist yet. Make sure your extraction cell ran successfully.")

Folders found inside metahuman_data:
['female_medium_average\\step_h7_f9_val_h1.08_f1.60_compressed.npz', 'female_medium_average\\step_h6_f9_val_h1.05_f1.60_compressed.npz', 'female_medium_average\\step_h4_f5_val_h0.98_f1.22_compressed.npz', 'female_medium_average\\step_h0_f2_val_h0.85_f0.94_compressed.npz', 'female_medium_average\\step_h5_f9_val_h1.02_f1.60_compressed.npz', 'female_medium_average\\step_h3_f0_val_h0.95_f0.75_compressed.npz', 'female_medium_average\\step_h0_f6_val_h0.85_f1.32_compressed.npz', 'female_medium_average\\step_h4_f1_val_h0.98_f0.84_compressed.npz', 'female_medium_average\\step_h2_f9_val_h0.92_f1.60_compressed.npz', 'female_medium_average\\step_h4_f2_val_h0.98_f0.94_compressed.npz', 'female_medium_average\\step_h8_f5_val_h1.12_f1.22_compressed.npz', 'female_medium_average\\step_h3_f9_val_h0.95_f1.60_compressed.npz', 'female_medium_average\\step_h1_f2_val_h0.88_f0.94_compressed.npz', 'female_medium_average\\step_h2_f3_val_h0.92_f1.03_compressed.npz', 'female_me

In [ ]:
%%writefile train_metahuman_autoencoder.py

import os
import glob
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR

# ==========================================
# 1. FIXED CONFIGURATION & HYPERPARAMETERS
# ==========================================
SEARCH_DIR = '/content/fmetahuman_data/'
NUM_VERTICES = 66993
INPUT_DIM = NUM_VERTICES * 3
LATENT_DIM = 32

BATCH_SIZE = 4
INITIAL_LR = 3e-4
EPOCHS = 80                   # Extended training budget for deep convergence
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Initializing Anharmonic Feature-Sharpening Pipeline on device: {DEVICE}")

# ==========================================
# 2. METAHUMAN DATASET LOADER
# ==========================================
class MetahumanMeshDataset(Dataset):
    def __init__(self, search_dir):
        search_pattern = os.path.join(search_dir, '**', '*.npz')
        self.filepaths = glob.glob(search_pattern, recursive=True)
        print(f"Successfully indexed {len(self.filepaths)} sample targets for training.")

        first_archive = np.load(self.filepaths[0])
        self.faces = first_archive['faces'].astype(np.int64)

        print("Calculating baseline anatomical anchor mesh...")
        all_meshes = []
        for i in range(min(10, len(self.filepaths))):
            arch = np.load(self.filepaths[i])
            all_meshes.append(arch['vertices'].flatten())
        self.mean_template = np.mean(all_meshes, axis=0).astype(np.float32)

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        data_path = self.filepaths[idx]
        archive = np.load(data_path)

        flat_mesh = archive['vertices'].flatten().astype(np.float32)
        height_target = float(archive['actual_height_scale'].item())
        fat_target = float(archive['actual_fat_scale'].item())

        biometric_targets = torch.tensor([height_target, fat_target], dtype=torch.float32)
        return torch.from_numpy(flat_mesh), biometric_targets

# ==========================================
# 3. HIGH-DEFINITION RESIDUAL CONVOLUTIONAL AUTOENCODER
# ==========================================
class HDMetahumanAutoencoder(nn.Module):
    def __init__(self, num_vertices, latent_dim, mean_template):
        super().__init__()
        self.num_vertices = num_vertices
        self.register_buffer('mean_mesh', torch.from_numpy(mean_template))

        self.encoder = nn.Sequential(
            nn.Linear(num_vertices * 3, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(128, latent_dim - 2)
        )

        self.decoder_dense = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, num_vertices * 4),
            nn.LeakyReLU(0.2, inplace=True)
        )

        self.decoder_conv = nn.Conv1d(
            in_channels=4, out_channels=3, kernel_size=5, padding=2, padding_mode='replicate'
        )

    def forward(self, x, biometrics):
        residual_latents = self.encoder(x)
        full_latents = torch.cat([biometrics, residual_latents], dim=1)

        features = self.decoder_dense(full_latents).view(-1, 4, self.num_vertices)
        predicted_deltas = self.decoder_conv(features).permute(0, 2, 1).contiguous()
        flat_deltas = predicted_deltas.view(-1, self.num_vertices * 3)

        return self.mean_mesh + flat_deltas

# ==========================================
# 4. VECTORIZED SURFACE NORMAL CONSISTENCY LOSS
# ==========================================
def compute_face_normals(vertices, faces):
    v0 = vertices[:, faces[:, 0], :]
    v1 = vertices[:, faces[:, 1], :]
    v2 = vertices[:, faces[:, 2], :]

    edge1 = v1 - v0
    edge2 = v2 - v0

    normals = torch.cross(edge1, edge2, dim=2)
    return nn.functional.normalize(normals, p=2, dim=2)

# ==========================================
# 5. EXECUTION LOOP WITH SCHEDULING
# ==========================================
if __name__ == '__main__':
    dataset = MetahumanMeshDataset(SEARCH_DIR)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
    faces_tensor = torch.from_numpy(dataset.faces).to(DEVICE)

    model = HDMetahumanAutoencoder(NUM_VERTICES, LATENT_DIM, dataset.mean_template).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=INITIAL_LR, weight_decay=1e-4)

    # Cosine Annealing scheduler smoothly reduces learning rate to let surface details lock in
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

    criterion_l1 = nn.L1Loss()

    print("\nExecuting crease-sharpening training sweep...")
    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0

        for batch_meshes, batch_targets in dataloader:
            batch_meshes = batch_meshes.to(DEVICE)
            batch_targets = batch_targets.to(DEVICE)

            optimizer.zero_grad()
            final_outputs = model(batch_meshes, batch_targets)

            # Coordinate delta penalty
            loss_coordinate = criterion_l1(final_outputs, batch_meshes)

            # Surface Normal penalty calculation
            pred_vertices = final_outputs.view(-1, NUM_VERTICES, 3)
            gt_vertices = batch_meshes.view(-1, NUM_VERTICES, 3)

            pred_normals = compute_face_normals(pred_vertices, faces_tensor)
            gt_normals = compute_face_normals(gt_vertices, faces_tensor)

            loss_normal = torch.mean(1.0 - nn.functional.cosine_similarity(pred_normals, gt_normals, dim=2))

            # AMPLIFIED CONFIGURATION: Weighting increased to 2.5x to aggressively force structural ridges
            loss = loss_coordinate + (2.5 * loss_normal)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * batch_meshes.size(0)

        scheduler.step()

        if epoch % 10 == 0 or epoch == 1:
            print(f"Epoch [{epoch:02d}/{EPOCHS}] -> Total Loss: {total_loss/len(dataset):.6f} | Normal Alignment Discrepancy: {loss_normal.item():.6f} | Current LR: {scheduler.get_last_lr()[0]:.1e}")

    torch.save(model.state_dict(), 'fmetahuman_autoencoder_trained.pth')
    np.save('mean_template.npy', dataset.mean_template)
    print("\nTraining complete! Fine surface variations successfully integrated.")

Writing train_metahuman_autoencoder.py


In [ ]:
!python train_metahuman_autoencoder.py

Initializing Anharmonic Feature-Sharpening Pipeline on device: cuda
Successfully indexed 100 sample targets for training.
Calculating baseline anatomical anchor mesh...

Executing crease-sharpening training sweep...
Epoch [01/80] -> Total Loss: 2.735096 | Normal Alignment Discrepancy: 0.964827 | Current LR: 3.0e-04
Epoch [10/80] -> Total Loss: 1.154528 | Normal Alignment Discrepancy: 0.380020 | Current LR: 2.9e-04
Epoch [20/80] -> Total Loss: 0.837956 | Normal Alignment Discrepancy: 0.278057 | Current LR: 2.6e-04
Epoch [30/80] -> Total Loss: 0.749871 | Normal Alignment Discrepancy: 0.240191 | Current LR: 2.1e-04
Epoch [40/80] -> Total Loss: 0.689554 | Normal Alignment Discrepancy: 0.224618 | Current LR: 1.5e-04
Epoch [50/80] -> Total Loss: 0.657458 | Normal Alignment Discrepancy: 0.212764 | Current LR: 9.3e-05
Epoch [60/80] -> Total Loss: 0.638215 | Normal Alignment Discrepancy: 0.207434 | Current LR: 4.5e-05
Epoch [70/80] -> Total Loss: 0.628847 | Normal Alignment Discrepancy: 0.20379

In [ ]:
%%writefile traverse_metahuman_latent.py

import os
import torch
import numpy as np
import torch.nn as nn

NUM_VERTICES = 66993
INPUT_DIM = NUM_VERTICES * 3
LATENT_DIM = 32
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class HDMetahumanAutoencoder(nn.Module):
    def __init__(self, num_vertices, latent_dim, mean_template):
        super().__init__()
        self.num_vertices = num_vertices
        self.register_buffer('mean_mesh', torch.from_numpy(mean_template))
        self.encoder = nn.Sequential(
            nn.Linear(num_vertices * 3, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(128, latent_dim - 2)
        )
        self.decoder_dense = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, num_vertices * 4),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.decoder_conv = nn.Conv1d(
            in_channels=4, out_channels=3, kernel_size=5, padding=2, padding_mode='replicate'
        )

if __name__ == '__main__':
    if os.path.exists('mean_template.npy') and os.path.exists('fmetahuman_autoencoder_trained.pth'):
        mean_template = np.load('mean_template.npy')
        model = HDMetahumanAutoencoder(NUM_VERTICES, LATENT_DIM, mean_template).to(DEVICE)
        model.load_state_dict(torch.load('fmetahuman_autoencoder_trained.pth', map_location=DEVICE))
        model.eval()

        out_dir = '/content/drive/MyDrive/PhD/Metahuman_project/'
        os.makedirs(out_dir, exist_ok=True)

        with torch.no_grad():
            residual_base = torch.zeros((1, LATENT_DIM - 2), dtype=torch.float32).to(DEVICE)

            # 1. Human Baseline (Height=1.0, Adiposity=1.0)
            latent_base = torch.cat([torch.tensor([[1.0, 1.0]], device=DEVICE), residual_base], dim=1)
            feat_base = model.decoder_dense(latent_base).view(-1, 4, NUM_VERTICES)
            deltas_base = model.decoder_conv(feat_base).permute(0, 2, 1).contiguous().view(-1, INPUT_DIM)
            mesh_base = (model.mean_mesh + deltas_base).cpu().numpy()[0]

            # Save format loop
            with open(os.path.join(out_dir, 'human_baseline.obj'), 'w') as f:
                for v in mesh_base.reshape(NUM_VERTICES, 3):
                    f.write(f"v {v[0]} {v[1]} {v[2]}\n")

            # 2. Tall Morph Axis Verification (Height=1.35, Adiposity=1.0)
            latent_tall = torch.cat([torch.tensor([[1.35, 1.0]], device=DEVICE), residual_base], dim=1)
            feat_tall = model.decoder_dense(latent_tall).view(-1, 4, NUM_VERTICES)
            deltas_tall = model.decoder_conv(feat_tall).permute(0, 2, 1).contiguous().view(-1, INPUT_DIM)
            mesh_tall = (model.mean_mesh + deltas_tall).cpu().numpy()[0]

            with open(os.path.join(out_dir, 'morph_height_tall.obj'), 'w') as f:
                for v in mesh_tall.reshape(NUM_VERTICES, 3):
                    f.write(f"v {v[0]} {v[1]} {v[2]}\n")

            # 3. Heavy Morph Axis Verification (Height=1.0, Adiposity=1.35)
            latent_heavy = torch.cat([torch.tensor([[1.0, 1.35]], device=DEVICE), residual_base], dim=1)
            feat_heavy = model.decoder_dense(latent_heavy).view(-1, 4, NUM_VERTICES)
            deltas_heavy = model.decoder_conv(feat_heavy).permute(0, 2, 1).contiguous().view(-1, INPUT_DIM)
            mesh_heavy = (model.mean_mesh + deltas_heavy).cpu().numpy()[0]

            with open(os.path.join(out_dir, 'morph_adiposity_heavy.obj'), 'w') as f:
                for v in mesh_heavy.reshape(NUM_VERTICES, 3):
                    f.write(f"v {v[0]} {v[1]} {v[2]}\n")

        print("\nAll pristine, surface-sharpened assets generated successfully!")

Writing traverse_metahuman_latent.py


In [ ]:
!python traverse_metahuman_latent.py


All pristine, surface-sharpened assets generated successfully!
